<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2025</strong></center>

### Cuerpo Docente:

- Profesores: Stefano Schiappacasse, Sebastián Tinoco
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Angelo Muñoz, Valentina Zúñiga

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Hayter Cortés Muñoz
- Nombre de alumno 2:


### **Link de repositorio de GitHub:** [Repositorio](https://github.com/HayterCortes/HayterCortes)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [16]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [17]:
import pandas as pd
import numpy as np
from datetime import datetime

# Cargar el dataset
df = pd.read_csv('sales.csv')

# Visualizar los primeros registros
print(df.head())

   id      date    city       lat      long     pop    shop        brand  \
0   0  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
1   1  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
2   2  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
3   3  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   
4   4  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   

  container capacity  price  quantity  
0     glass    500ml   0.96     13280  
1   plastic    1.5lt   2.86      6727  
2       can    330ml   0.87      9848  
3     glass    500ml   1.00     20050  
4       can    330ml   0.39     25696  


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [18]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")

# Definir características (X) y objetivo (y)
# Exclusión de 'id' por ser un identificador y 'quantity' por ser el objetivo.
# También 'date' por ahora
X = df.drop(columns=['id', 'date', 'quantity'])
y = df['quantity']

# Fijar una semilla para la reproducibilidad
random_seed = 42

# Primera división: separar el conjunto de test (10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=random_seed
)

# Segunda división: separar el conjunto de entrenamiento (70% del original) y validación (20% del original)
# train_size será 0.70 / (1.0 - 0.10) = 0.70 / 0.90 = 7/9
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.20 / (1-0.10)), random_state=random_seed # 0.20 / 0.90 = 2/9
)

# Imprimir las formas de los conjuntos para verificar
print(f"Forma de X_train: {X_train.shape}, Forma de y_train: {y_train.shape}")
print(f"Forma de X_val: {X_val.shape}, Forma de y_val: {y_val.shape}")
print(f"Forma de X_test: {X_test.shape}, Forma de y_test: {y_test.shape}")

# Mostrar las primeras filas de X_train para verificar las columnas
print("\nPrimeras filas de X_train:")
print(X_train.head())


Forma de X_train: (5218, 9), Forma de y_train: (5218,)
Forma de X_val: (1492, 9), Forma de y_val: (1492,)
Forma de X_test: (746, 9), Forma de y_test: (746,)

Primeras filas de X_train:
           city       lat      long     pop    shop         brand container  \
6907      Patra  38.24444  21.73444  168034  shop_6   kinder-cola   plastic   
2687      Patra  38.24444  21.73444  167242  shop_6  orange-power     glass   
6451     Larisa  39.63689  22.41761  144651  shop_5    adult-cola     glass   
6028     Athens  37.97945  23.71622  665871  shop_1  orange-power   plastic   
3660  Irakleion  35.32787  25.14341  138560  shop_2        gazoza   plastic   

     capacity  price  
6907    1.5lt   2.93  
2687    500ml   0.67  
6451    500ml   1.18  
6028    1.5lt   1.71  
3660    1.5lt   1.03  


In [ ]:
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split

# Función para extraer características de la fecha
def extract_date_features(dataframe_con_col_date):
    date_series = pd.to_datetime(dataframe_con_col_date['date'], format='%d/%m/%y')
    
    date_features_df = pd.DataFrame({
        'day': date_series.dt.day.astype('category'),
        'month': date_series.dt.month.astype('category'),
        'year': date_series.dt.year.astype('category')
    })
    return date_features_df

# Crear el FunctionTransformer
date_transformer = FunctionTransformer(extract_date_features)

# Redefinir X e y para que 'date' esté en X
# y pueda ser procesada por el pipeline completo.
X_full = df.drop(columns=['id', 'quantity']) 
y_full = df['quantity']

X_temp_full, X_test_full, y_temp_full, y_test = train_test_split(
    X_full, y_full, test_size=0.10, random_state=random_seed
)

X_train_full, X_val_full, y_train, y_val = train_test_split(
    X_temp_full, y_temp_full, test_size=(0.20 / (1-0.10)), random_state=random_seed 
)

# Imprimir las formas de los conjuntos para verificar
print(f"Forma de X_train_full: {X_train_full.shape}, Forma de y_train: {y_train.shape}")
print(f"Forma de X_val_full: {X_val_full.shape}, Forma de y_val: {y_val.shape}")
print(f"Forma de X_test_full: {X_test_full.shape}, Forma de y_test: {y_test.shape}")

# Ahora X_train_full, X_val_full, X_test_full contienen la columna 'date'
print("\nPrimeras filas de X_train_full (con 'date'):")
print(X_train_full.head())

# Primeras filas de X_val_full y X_test_full para asegurar que 'date' está presente
print("\nPrimeras filas de X_val_full (con 'date'):")
print(X_val_full.head())

print("\nPrimeras filas de X_test_full (con 'date'):")
print(X_test_full.head())

Forma de X_train_full: (5218, 10), Forma de y_train: (5218,)
Forma de X_val_full: (1492, 10), Forma de y_val: (1492,)
Forma de X_test_full: (746, 10), Forma de y_test: (746,)

Primeras filas de X_train_full (con 'date'):
          date       city       lat      long     pop    shop         brand  \
6907  30/06/18      Patra  38.24444  21.73444  168034  shop_6   kinder-cola   
2687  31/07/14      Patra  38.24444  21.73444  167242  shop_6  orange-power   
6451  31/01/18     Larisa  39.63689  22.41761  144651  shop_5    adult-cola   
6028  30/09/17     Athens  37.97945  23.71622  665871  shop_1  orange-power   
3660  30/06/15  Irakleion  35.32787  25.14341  138560  shop_2        gazoza   

     container capacity  price  
6907   plastic    1.5lt   2.93  
2687     glass    500ml   0.67  
6451     glass    500ml   1.18  
6028   plastic    1.5lt   1.71  
3660   plastic    1.5lt   1.03  

Primeras filas de X_val_full (con 'date'):
          date       city       lat      long     pop    shop 

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# 1. Identificar tipos de columnas en X_train_full
numerical_cols = ['lat', 'long', 'pop', 'price']
categorical_cols_original = ['city', 'shop', 'brand', 'container', 'capacity']

# 2. Crear un pipeline para el preprocesamiento de la columna 'date'
date_pipeline = Pipeline([
    ('extract_date_features', date_transformer),
    ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Crear el ColumnTransformer principal
preprocessor = ColumnTransformer(
    transformers=[
        ('date_processing', date_pipeline, ['date']),
        # Corregido: Se elimina min_frequency=0.0 para usar el valor por defecto (None)
        ('categorical_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_original),
        ('numerical_scaling', StandardScaler(), numerical_cols)
    ],
    remainder='drop'
)

# Preprocesador con los datos de entrenamiento 
print("Ajustando el preprocesador con X_train_full...")
X_train_processed = preprocessor.fit_transform(X_train_full)
print(f"Forma de X_train_processed: {X_train_processed.shape}")
print("Primeras filas de X_train_processed:")
print(X_train_processed.head())

# X_val_full (sin re-ajustar)
X_val_processed = preprocessor.transform(X_val_full)
print(f"\nForma de X_val_processed: {X_val_processed.shape}")
print("Primeras filas de X_val_processed (primeras 5):") 
print(X_val_processed.head())


# X_test_full (sin re-ajustar)
X_test_processed = preprocessor.transform(X_test_full)
print(f"\nForma de X_test_processed: {X_test_processed.shape}")
print("Primeras filas de X_test_processed (primeras 5):") 
print(X_test_processed.head())

Ajustando el preprocesador con X_train_full...
Forma de X_train_processed: (5218, 49)
Primeras filas de X_train_processed:
      date_processing__day_28  date_processing__day_29  \
6907                      0.0                      0.0   
2687                      0.0                      0.0   
6451                      0.0                      0.0   
6028                      0.0                      0.0   
3660                      0.0                      0.0   

      date_processing__day_30  date_processing__day_31  \
6907                      1.0                      0.0   
2687                      0.0                      1.0   
6451                      0.0                      1.0   
6028                      1.0                      0.0   
3660                      1.0                      0.0   

      date_processing__month_1  date_processing__month_2  \
6907                       0.0                       0.0   
2687                       0.0                       0.0   

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

# 1. Identificar tipos de columnas en X_train_full
numerical_cols = ['lat', 'long', 'pop', 'price']
categorical_cols_original = ['city', 'shop', 'brand', 'container', 'capacity']

# 2. Crear un pipeline para el preprocesamiento de la columna 'date'
date_pipeline = Pipeline([
    ('extract_date_features', date_transformer), 
    ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Crear el ColumnTransformer principal
preprocessor_for_pipeline = ColumnTransformer(
    transformers=[
        ('date_processing', date_pipeline, ['date']),
        ('categorical_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_original),
        ('numerical_scaling', StandardScaler(), numerical_cols)
    ],
    remainder='drop'
)

# 4. Crear el pipeline con el preprocesador y DummyRegressor
pipeline_dummy = Pipeline([
    ('preprocessor', preprocessor_for_pipeline), # Paso de preprocesamiento
    ('regressor', DummyRegressor(strategy='mean')) # Modelo DummyRegressor
])

# Mostrar el pipeline para verificar su estructura
print("Pipeline con DummyRegressor creado:")
print(pipeline_dummy)

Pipeline con DummyRegressor creado:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('date_processing',
                                                  Pipeline(steps=[('extract_date_features',
                                                                   FunctionTransformer(func=<function extract_date_features at 0x000002211FC64220>)),
                                                                  ('ohe_date_features',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['date']),
                                                 ('categorical_onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
            

In [ ]:
from sklearn.metrics import mean_absolute_error

# 1. Entrenar el pipeline_dummy
print("Entrenando el pipeline_dummy...")
pipeline_dummy.fit(X_train_full, y_train)
print("Pipeline_dummy entrenado.")

# 2. Realizar predicciones en el conjunto de validación
y_pred_dummy_val = pipeline_dummy.predict(X_val_full)

# 3. Calcular el Mean Absolute Error (MAE)
mae_dummy_val = mean_absolute_error(y_val, y_pred_dummy_val)

print(f"\nMAE del DummyRegressor en el conjunto de validación: {mae_dummy_val:.2f}")

Entrenando el pipeline_dummy...
Pipeline_dummy entrenado.

MAE del DummyRegressor en el conjunto de validación: 13543.96


**Interpretación del MAE del DummyRegressor (13543.96) en el contexto del negocio:**

Este valor significa que, si simplemente predijéramos la cantidad de venta promedio (que es lo que hace `DummyRegressor(strategy='mean')`) para cualquier producto, en cualquier fecha y tienda, nuestro error promedio sería de aproximadamente 13,544 unidades de producto. Es decir, en promedio, nuestras predicciones estarían equivocadas por 13,544 unidades (ya sea por encima o por debajo de la cantidad real vendida).

Considerando que se trata de la demanda de una productora de bebidas, esta desviación es probablemente muy alta. Si "unidades" se refiere a botellas individuales o latas, un error de más de 13,000 unidades por predicción (que podría ser para un producto específico en un día y tienda dados) podría llevar a:
* **Subestimación:** Pérdida significativa de ventas por falta de stock si la demanda real es mayor.
* **Sobrestimación:** Costos excesivos de inventario, almacenamiento y potencial desperdicio de producto si la demanda real es menor.

Este MAE de 13543.96 establece el **baseline de error**. Cualquier modelo más sofisticado debe tener como objetivo reducir significativamente este número para ser considerado útil.

In [ ]:
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# 1. Crear el pipeline con el preprocesador y XGBRegressor (parámetros por defecto)
pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor_for_pipeline), 
    ('regressor', xgb.XGBRegressor(random_state=random_seed)) 
])

# 2. Entrenar el pipeline_xgb
print("Entrenando el pipeline_xgb...")
pipeline_xgb.fit(X_train_full, y_train)
print("Pipeline_xgb entrenado.")

# 3. Realizar predicciones en el conjunto de validación
y_pred_xgb_val = pipeline_xgb.predict(X_val_full)

# 4. Calcular el Mean Absolute Error (MAE)
mae_xgb_val = mean_absolute_error(y_val, y_pred_xgb_val)

print(f"\nMAE del DummyRegressor en el conjunto de validación: {mae_dummy_val:.2f}") 
print(f"MAE del XGBRegressor (default) en el conjunto de validación: {mae_xgb_val:.2f}")

# 5. Comparar y analizar
if mae_xgb_val < mae_dummy_val:
    print("\nXGBRegressor es MEJOR que DummyRegressor.")
    print(f"La reducción en MAE es de: {(mae_dummy_val - mae_xgb_val):.2f} unidades.")
else:
    print("\nXGBRegressor NO es mejor (o es peor) que DummyRegressor con los parámetros por defecto.")

Entrenando el pipeline_xgb...
Pipeline_xgb entrenado.

MAE del DummyRegressor en el conjunto de validación: 13543.96
MAE del XGBRegressor (default) en el conjunto de validación: 2404.40

XGBRegressor es MEJOR que DummyRegressor.
La reducción en MAE es de: 11139.56 unidades.


In [ ]:
import joblib

# Nombre de los archivos para guardar los modelos
path_dummy_model = 'dummy_model_baseline.pkl'
path_xgb_model = 'xgb_model_baseline.pkl'

# Guardar el pipeline_dummy
print(f"Guardando pipeline_dummy en {path_dummy_model}...")
joblib.dump(pipeline_dummy, path_dummy_model)
print("pipeline_dummy guardado.")

# Guardar el pipeline_xgb
print(f"\nGuardando pipeline_xgb en {path_xgb_model}...")
joblib.dump(pipeline_xgb, path_xgb_model)
print("pipeline_xgb guardado.")

Guardando pipeline_dummy en dummy_model_baseline.pkl...
pipeline_dummy guardado.

Guardando pipeline_xgb en xgb_model_baseline.pkl...
pipeline_xgb guardado.


## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [ ]:
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error
import numpy as np

def get_date_feature_names_out(function_transformer_instance, input_features=None):
    return ['day', 'month', 'year']

# Redefinir date_transformer con feature_names_out
date_transformer_corrected = FunctionTransformer(
    extract_date_features, # Tu función original extract_date_features
    feature_names_out=get_date_feature_names_out
)

# Redefinir date_pipeline con el date_transformer_corrected
date_pipeline_corrected = Pipeline([
    ('extract_date_features', date_transformer_corrected),
    ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Redefinir preprocessor_corrected
preprocessor_corrected = ColumnTransformer(
    transformers=[
        ('date_processing', date_pipeline_corrected, ['date']),
        ('categorical_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_original),
        ('numerical_scaling', StandardScaler(), numerical_cols)
    ],
    remainder='drop'
)

# 1. Ajustar el preprocesador corregido y obtener los nombres de las características
print("Ajustando el preprocesador corregido con X_train_full...")
preprocessor_corrected.fit(X_train_full, y_train) # Ajustar con X_train_full y y_train
feature_names_processed = preprocessor_corrected.get_feature_names_out()
print(f"Nombres de características procesadas obtenidos (total: {len(feature_names_processed)}). Ejemplo: {feature_names_processed[:3]}")


# 2. Identificar el nombre exacto de la columna de precio procesada y su índice
price_column_processed_name = 'numerical_scaling__price'

try:
    price_column_index = list(feature_names_processed).index(price_column_processed_name)
    print(f"Columna de precio procesada: '{price_column_processed_name}' encontrada en el índice: {price_column_index}")

    # 3. Crear la tupla de restricciones monotónicas
    n_features = len(feature_names_processed)
    constraints = [0] * n_features
    constraints[price_column_index] = -1
    monotonic_constraints_tuple = tuple(constraints)
    print(f"Restricciones monótonas (extracto cerca del precio): {monotonic_constraints_tuple[max(0,price_column_index-2):price_column_index+3]}")

    # 4. Definir el XGBRegressor con restricciones monótonas
    xgb_regressor_monotonic = xgb.XGBRegressor(
        random_state=random_seed,
        monotone_constraints=monotonic_constraints_tuple
    )

    # 5. Crear el nuevo pipeline usando el preprocesador corregido
    pipeline_xgb_monotonic = Pipeline([
        ('preprocessor', preprocessor_corrected),
        ('regressor', xgb_regressor_monotonic)
    ])

    # 6. Entrenar el pipeline con restricciones monótonas
    print("\nEntrenando el pipeline_xgb_monotonic...")
    pipeline_xgb_monotonic.fit(X_train_full, y_train)
    print("Pipeline_xgb_monotonic entrenado.")

    # 7. Realizar predicciones y calcular MAE
    y_pred_xgb_monotonic_val = pipeline_xgb_monotonic.predict(X_val_full)
    mae_xgb_monotonic_val = mean_absolute_error(y_val, y_pred_xgb_monotonic_val)

    print(f"\nMAE del XGBRegressor (default) en validación: {mae_xgb_val:.2f}")
    print(f"MAE del XGBRegressor (con restricción monótona en precio) en validación: {mae_xgb_monotonic_val:.2f}")

except ValueError as ve:
    print(f"Error de Valor: {ve}")
    print(f"La columna '{price_column_processed_name}' podría no encontrarse en las características procesadas.")

except NameError as ne:
    print(f"Error de Nombre: Una variable no está definida.")

except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

Ajustando el preprocesador corregido con X_train_full...
Nombres de características procesadas obtenidos (total: 49). Ejemplo: ['date_processing__day_28' 'date_processing__day_29'
 'date_processing__day_30']
Columna de precio procesada: 'numerical_scaling__price' encontrada en el índice: 48
Restricciones monótonas (extracto cerca del precio): (0, 0, -1)

Entrenando el pipeline_xgb_monotonic...
Pipeline_xgb_monotonic entrenado.

MAE del XGBRegressor (default) en validación: 2404.40
MAE del XGBRegressor (con restricción monótona en precio) en validación: 2524.87


**2.3 ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo?**

**Análisis del cambio en el error:**
Al incluir la restricción monótona negativa para el precio, el MAE en el conjunto de validación **aumentó** de 2404.40 a 2524.87. Esto es un incremento de aproximadamente 120.47 unidades en el error.

Por lo tanto, en términos de precisión predictiva medida por el MAE en este conjunto de validación, el modelo con la restricción monótona **funcionó ligeramente peor** que el modelo XGBoost sin esta restricción.

**¿Tenía razón su amigo?**
El amigo, aficionado a la economía, sugirió que la demanda guarda una relación inversa con el precio del producto. Esta es la base de la "ley de la demanda" en economía, que es un principio generalmente aceptado y observado. Así que, **teóricamente, el amigo tenía una base sólida para su sugerencia.**

Sin embargo, el hecho de que forzar esta restricción en el modelo haya empeorado (aunque sea ligeramente) el MAE sugiere algunas posibilidades:

1.  **La relación no es estrictamente monótona en todos los casos dentro de este dataset específico:** Si bien la tendencia general podría ser que a mayor precio menor demanda, puede haber segmentos de productos, tiendas, periodos de tiempo o situaciones (promociones, productos de lujo/Veblen, etc.) donde esta relación no se cumple estrictamente o es más compleja. El modelo sin restricciones podría estar capturando estas excepciones o matices.
2.  **La restricción es demasiado rígida:** Al forzar la monotonicidad, podríamos estar impidiendo que el modelo se ajuste óptimamente a los datos, incluso si la tendencia general es correcta. El modelo podría necesitar cierta flexibilidad que la restricción le quita.
3.  **Interacción con otras características o el preprocesamiento:** La forma en que el precio fue preprocesado (escalado con `StandardScaler`) o cómo interactúa con otras características podría influir en el efecto de la restricción.
4.  **Variabilidad del dataset:** El ligero empeoramiento podría estar dentro de la variabilidad normal del rendimiento del modelo, aunque la diferencia es notable.

**En conclusión:** Si bien la intuición económica del amigo es válida en general, en este caso particular y con la implementación actual, forzar explícitamente esta relación no mejoró la capacidad predictiva del modelo XGBoost según el MAE. El modelo sin restricciones fue capaz de encontrar un mejor ajuste (menor error) a los datos de validación. No obstante, la idea de incorporar conocimiento del dominio es valiosa, y a veces estas restricciones pueden ayudar a la generalización o interpretabilidad, incluso si no mejoran marginalmente una métrica específica.

In [ ]:
import joblib
path_xgb_monotonic_model = 'xgb_model_monotonic.pkl'

# Guardar el pipeline_xgb_monotonic
print(f"\nGuardando pipeline_xgb_monotonic en {path_xgb_monotonic_model}...")
joblib.dump(pipeline_xgb_monotonic, path_xgb_monotonic_model)
print("pipeline_xgb_monotonic guardado.")


Guardando pipeline_xgb_monotonic en xgb_model_monotonic.pkl...
pipeline_xgb_monotonic guardado.


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error
import joblib 
import time # Para el timeout

optuna.logging.set_verbosity(optuna.logging.WARNING) # Silenciar logs de Optuna por trial

def objective(trial):
    # Para OneHotEncoder: min_frequency en rango flotante (0.0, 1.0) excluyendo 0 y 1.
    ohe_min_freq = trial.suggest_float('ohe_min_frequency', 1e-7, 1.0 - 1e-7) # Ajustado para estar dentro de (0,1)

    # Para XGBRegressor
    xgb_params = {
        'random_state': random_seed,
        'learning_rate': trial.suggest_float('xgb_learning_rate', 0.001, 0.1, log=True),
        'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 1000),
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
        'max_leaves': trial.suggest_int('xgb_max_leaves', 0, 100), # 0 significa sin límite
        'min_child_weight': trial.suggest_int('xgb_min_child_weight', 1, 5),
        'reg_alpha': trial.suggest_float('xgb_reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('xgb_reg_lambda', 0.0, 1.0),
        'objective': 'reg:squarederror', # Explicit objective
        'n_jobs': -1 # Usar todos los cores disponibles
    }

    # Construir el Preprocesador
    date_pipeline_trial = Pipeline([
        ('extract_date_features', date_transformer_corrected),
        ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=ohe_min_freq))
    ])

    preprocessor_trial = ColumnTransformer(
        transformers=[
            ('date_processing', date_pipeline_trial, ['date']),
            ('categorical_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=ohe_min_freq), categorical_cols_original),
            ('numerical_scaling', StandardScaler(), numerical_cols)
        ],
        remainder='drop'
    )

    model_trial = xgb.XGBRegressor(**xgb_params)

    full_pipeline_trial = Pipeline([
        ('preprocessor', preprocessor_trial),
        ('regressor', model_trial)
    ])

    full_pipeline_trial.fit(X_train_full, y_train)
    predictions = full_pipeline_trial.predict(X_val_full)
    mae = mean_absolute_error(y_val, predictions)

    trial.set_user_attr("trained_pipeline", full_pipeline_trial)
    return mae

# --- Crear el estudio de Optuna y ejecutar la optimización ---

# 1. Configurar el sampler TPESampler con una semilla para reproducibilidad
sampler = TPESampler(seed=random_seed)

# 2. Crear el estudio, especificando la dirección de optimización y el sampler
study_optuna = optuna.create_study(direction='minimize', sampler=sampler)

# 3. Definir el tiempo de entrenamiento (5 minutos = 300 segundos)
timeout_seconds = 300

print(f"Iniciando optimización con Optuna por {timeout_seconds} segundos...")
start_time = time.time()
study_optuna.optimize(objective, timeout=timeout_seconds)
end_time = time.time()
print(f"Optimización completada en {end_time - start_time:.2f} segundos.")

# --- Reportar resultados de la optimización (Punto 1.3.3 del lab) ---
print(f"\nNúmero de trials completados: {len(study_optuna.trials)}")
print(f"Mejor MAE obtenido en validación: {study_optuna.best_value:.2f}")
print("Mejores hiperparámetros encontrados:")
for key, value in study_optuna.best_params.items():
    print(f"  {key}: {value}")

# Recuperar el mejor pipeline entrenado
best_pipeline_optuna = study_optuna.best_trial.user_attrs["trained_pipeline"]
print("\nMejor pipeline recuperado.")

# Comparación con la sección anterior (XGBoost default MAE)
if 'mae_xgb_val' in globals() or 'mae_xgb_val' in locals():
    print(f"\nMAE del XGBoost con parámetros por default (sección anterior): {mae_xgb_val:.2f}")
    if study_optuna.best_value < mae_xgb_val:
        print("Optuna ENCONTRÓ un mejor modelo.")
        print(f"Reducción en MAE respecto al XGBoost default: {(mae_xgb_val - study_optuna.best_value):.2f}")
    else:
        print("Optuna NO encontró un modelo significativamente mejor que el XGBoost default con estos rangos/tiempo.")
else:
    print("\nVariable 'mae_xgb_val' (MAE del XGBoost default) no encontrada para comparación.")

Iniciando optimización con Optuna por 300 segundos...
Optimización completada en 300.68 segundos.

Número de trials completados: 240
Mejor MAE obtenido en validación: 1931.11
Mejores hiperparámetros encontrados:
  ohe_min_frequency: 0.06087978788500909
  xgb_learning_rate: 0.09077657830572917
  xgb_n_estimators: 800
  xgb_max_depth: 10
  xgb_max_leaves: 97
  xgb_min_child_weight: 3
  xgb_reg_alpha: 0.16572669113586125
  xgb_reg_lambda: 0.7312898025623251

Mejor pipeline recuperado.

MAE del XGBoost con parámetros por default (sección anterior): 2404.40
Optuna ENCONTRÓ un mejor modelo.
Reducción en MAE respecto al XGBoost default: 473.29


Resumen de la optimización:
* **Tiempo de optimización**: ~300.34 segundos (los 5 minutos establecidos).
* **Número de trials completados**: 237.
* **Mejor MAE obtenido en validación**: **1931.11**.
* **Mejores hiperparámetros encontrados**:
    * `ohe_min_frequency`: 0.06087978788500909
    * `xgb_learning_rate`: 0.09077657830572917
    * `xgb_n_estimators`: 800
    * `xgb_max_depth`: 10
    * `xgb_max_leaves`: 97
    * `xgb_min_child_weight`: 3
    * `xgb_reg_alpha`: 0.16572669113586125
    * `xgb_reg_lambda`: 0.7312898025623251
* **Comparación**:
    * MAE del XGBoost con parámetros por default (sección anterior): 2404.40
    * Optuna **ENCONTRÓ un mejor modelo**, logrando una reducción en MAE de **473.29** unidades.

**Punto 1.3.3 (Análisis): ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto?**

Los resultados han mejorado significativamente. El MAE se redujo de 2404.40 (XGBoost con parámetros por defecto) a **1931.11** con los hiperparámetros encontrados por Optuna. Esta es una mejora considerable y demuestra el valor de la optimización de hiperparámetros.

**¿A qué se puede deber esta mejora?**
1.  **Búsqueda Sistemática**: Optuna (con `TPESampler`) realiza una búsqueda más inteligente que una simple prueba manual o una búsqueda en grilla (grid search) exhaustiva. Explora el espacio de hiperparámetros de manera eficiente, aprendiendo de los trials anteriores para enfocarse en regiones más prometedoras.
2.  **Ajuste Específico al Dataset**: Los parámetros por defecto de `XGBoost` (y `OneHotEncoder`) están diseñados para funcionar razonablemente bien en una amplia gama de problemas, pero raramente son óptimos para un dataset específico. Optuna encontró una combinación de valores que se ajusta mejor a las características y la complejidad de *tus* datos de ventas.
3.  **Interacción de Hiperparámetros**: Los hiperparámetros a menudo interactúan entre sí. Optuna puede descubrir combinaciones efectivas que serían difíciles de encontrar manualmente. Por ejemplo, un `learning_rate` más alto podría funcionar bien con un `n_estimators` diferente, o `max_depth` podría interactuar con `min_child_weight`.
4.  **Optimización del Preprocesamiento**: Es importante destacar que no solo optimizamos `XGBoost`, sino también el `min_frequency` del `OneHotEncoder`. Un valor de `ohe_min_frequency` de ~0.06 significa que las categorías que aparecen en menos del 6% de las muestras de entrenamiento para una característica dada se agruparán en una categoría "infrecuente" (si `handle_unknown='infrequent_if_exist'` estuviera activo y se configurara `max_categories` o similar, o si se combina con `max_categories`). En nuestro caso con `handle_unknown='ignore'`, este `min_frequency` ayuda a reducir la dimensionalidad y el ruido al no crear columnas separadas para categorías muy raras, lo que puede mejorar la generalización del modelo.

**Punto 1.3.4: Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados?**

**Para `OneHotEncoder`:**

* **`ohe_min_frequency` (valor óptimo: ~0.0609)**:
    * **Rol**: Si es un float, especifica la fracción mínima de muestras que una categoría debe tener para ser codificada como su propia característica one-hot. Las categorías con una frecuencia menor a este umbral pueden ser tratadas de acuerdo a `handle_unknown` o `max_categories` (en nuestro caso, con `handle_unknown='ignore'` y sin `max_categories`, las categorías infrecuentes que no se ven en `fit` se ignorarán, y las que se ven pero son infrecuentes *sí* se codifican a menos que `max_categories` las pode. Con `min_frequency` > 0, las categorías raras del set de entrenamiento que no alcanzan este umbral se tratan como si fueran desconocidas/infrecuentes). El valor encontrado (~6%) sugiere que agrupar categorías muy raras fue beneficioso.
    * **Rango `(1e-7, 1.0 - 1e-7)`**: Este rango tiene sentido. Permite explorar desde no agrupar casi nada (valor muy pequeño) hasta agrupar categorías bastante comunes (cerca de 1.0, aunque valores tan altos suelen ser contraproducentes). El valor óptimo de ~0.06 indica que un umbral moderado fue útil.

**Para `XGBRegressor`:**

* **`learning_rate` (eta) (valor óptimo: ~0.0908)**:
    * **Rol**: Controla el tamaño del paso en cada iteración de boosting, reduciendo la contribución de cada árbol. Valores más bajos hacen que el modelo sea más robusto a la sobreajuste, pero requieren más árboles (`n_estimators`).
    * **Rango `(0.001, 0.1)` logarítmico**: Es un rango estándar y muy sensato. Los learning rates suelen variar en órdenes de magnitud. El valor encontrado (~0.09) está en el extremo superior de este rango, lo que sugiere que el modelo pudo aprender relativamente rápido sin sobreajustar demasiado, especialmente con la regularización y un número adecuado de estimadores.

* **`n_estimators` (valor óptimo: 800)**:
    * **Rol**: El número total de árboles (rondas de boosting) a construir. Más árboles pueden mejorar el rendimiento, pero también pueden llevar a sobreajuste si no se controlan con otros parámetros (como `learning_rate` o early stopping).
    * **Rango `(50, 1000)`**: Es un buen rango. 50 es un mínimo razonable para que el boosting tenga efecto, y 1000 permite modelos bastante complejos. 800 árboles es un número considerable, lo que, junto con un `learning_rate` más alto, indica un modelo potente.

* **`max_depth` (valor óptimo: 10)**:
    * **Rol**: La profundidad máxima de cada árbol individual. Árboles más profundos pueden capturar interacciones más complejas, pero también son más propensos al sobreajuste.
    * **Rango `(3, 10)`**: Es un rango común. El valor óptimo de 10 sugiere que el modelo se benefició de árboles bastante profundos para capturar la complejidad de los datos.

* **`max_leaves` (valor óptimo: 97)**:
    * **Rol**: El número máximo de nodos hoja en cada árbol. Si es 0, no hay límite (aparte de `max_depth`). Controla la complejidad del árbol de una manera diferente a `max_depth`. A veces se usa en lugar de `max_depth`.
    * **Rango `(0, 100)`**: Permite explorar desde árboles sin límite de hojas (cuando es 0) hasta árboles con un número considerable de hojas. Un valor de 97 con `max_depth` de 10 es consistente con árboles complejos.

* **`min_child_weight` (valor óptimo: 3)**:
    * **Rol**: Suma mínima de pesos de instancia (hessian) necesarios en un nodo hijo. Si el paso de partición de un árbol resulta en un nodo hoja con una suma de pesos de instancia menor que `min_child_weight`, entonces el proceso de construcción dejará de particionar. Ayuda a controlar el sobreajuste. Valores más altos hacen al algoritmo más conservador.
    * **Rango `(1, 5)`**: Rango típico para este parámetro. Un valor de 3 indica una regularización moderada en este aspecto.

* **`reg_alpha` (L1 regularization) (valor óptimo: ~0.1657)**:
    * **Rol**: Término de regularización L1 sobre los pesos de las hojas. Puede llevar a que algunos pesos de las hojas sean cero, realizando una especie de selección de características. Ayuda a prevenir el sobreajuste.
    * **Rango `(0.0, 1.0)`**: Es un rango estándar. El valor encontrado indica que una pequeña cantidad de regularización L1 fue útil.

* **`reg_lambda` (L2 regularization) (valor óptimo: ~0.7313)**:
    * **Rol**: Término de regularización L2 sobre los pesos de las hojas. Hace que los pesos de las hojas sean más pequeños y el modelo más suave. Ayuda a prevenir el sobreajuste.
    * **Rango `(0.0, 1.0)`**: Es un rango estándar. El valor encontrado, más alto que `reg_alpha`, sugiere que la regularización L2 tuvo un papel más prominente en este modelo óptimo.

**En general, los rangos de optimización proporcionados son sensatos y cubren valores comúnmente efectivos para estos hiperparámetros.** Los valores óptimos encontrados por Optuna parecen coherentes: un modelo con un número relativamente alto de árboles, árboles profundos y con bastantes hojas, un learning rate no demasiado pequeño, y una regularización L1 y L2 moderada.

In [28]:
# Nombre del archivo para guardar el mejor modelo de Optuna
path_optuna_best_model = 'xgb_model_optuna_best.pkl'

# Guardar el pipeline_xgb_monotonic
print(f"\nGuardando el mejor pipeline de Optuna en {path_optuna_best_model}...")
joblib.dump(best_pipeline_optuna, path_optuna_best_model)
print(f"Mejor pipeline de Optuna guardado en {path_optuna_best_model}.")


Guardando el mejor pipeline de Optuna en xgb_model_optuna_best.pkl...
Mejor pipeline de Optuna guardado en xgb_model_optuna_best.pkl.


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación


**¿Qué es prunning?**

En el contexto de la optimización de hiperparámetros con Optuna, la poda (pruning) es una técnica que consiste en **detener tempranamente los "trials" (pruebas de combinaciones de hiperparámetros) que no parecen prometedores**.
En lugar de dejar que cada trial complete todo su entrenamiento (lo que puede ser costoso si el modelo es complejo o el dataset es grande), el "pruner" (el componente de Optuna que decide si podar) observa el rendimiento del trial en puntos intermedios (por ejemplo, después de un cierto número de épocas o, en el caso de XGBoost, después de un cierto número de árboles construidos).
Si el rendimiento intermedio de un trial es significativamente peor que el de otros trials (ya sea completados o aquellos que han alcanzado una etapa similar), Optuna puede decidir "podar" ese trial, es decir, detenerlo antes de que termine.

**¿De qué forma debería impactar en el entrenamiento (del proceso de optimización)?**
1.  **Mayor Eficiencia**: El principal impacto es un aumento en la eficiencia del proceso de búsqueda de hiperparámetros. Al descartar rápidamente los trials malos, se ahorra tiempo y recursos computacionales.
2.  **Más Trials Explorados**: Con un presupuesto de tiempo fijo, la poda permite que Optuna evalúe un mayor número de combinaciones diferentes de hiperparámetros, ya que no pierde tanto tiempo en las que claramente no van a funcionar bien.
3.  **Potencial para Encontrar Mejores Modelos Más Rápido**: Al explorar más el espacio de búsqueda, aumenta la probabilidad de encontrar una combinación de hiperparámetros que resulte en un mejor modelo dentro del tiempo asignado.
4.  **Riesgo de Poda Prematura (Pequeño)**: Existe un pequeño riesgo de que un trial que es podado tempranamente pudiera haber mejorado significativamente más adelante en su entrenamiento. Sin embargo, los algoritmos de poda están diseñados para minimizar este riesgo, basándose en el rendimiento comparativo.

In [29]:
!pip install optuna-integration[xgboost]

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.integration import XGBoostPruningCallback
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_with_pruning(trial):
    # 1) Hiperparámetros
    ohe_min_freq = trial.suggest_float('ohe_min_frequency', 1e-7, 1.0 - 1e-7)
    params = {
        'learning_rate': trial.suggest_float('xgb_learning_rate', 0.001, 0.1, log=True),
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
        'max_leaves': trial.suggest_int('xgb_max_leaves', 0, 100),
        'min_child_weight': trial.suggest_int('xgb_min_child_weight', 1, 5),
        'reg_alpha': trial.suggest_float('xgb_reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('xgb_reg_lambda', 0.0, 1.0),
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',  # <-- clave para el pruning con MAE
        'random_state': random_seed,
        'n_jobs': -1
    }

    # 2) Construir el preprocesador
    date_pipeline = Pipeline([
        ('extract_date_features', date_transformer_corrected),
        ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=ohe_min_freq))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('date', date_pipeline, ['date']),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=ohe_min_freq), categorical_cols_original),
            ('num', StandardScaler(), numerical_cols)
        ],
        remainder='drop'
    )

    # 3) Preprocesar manualmente para DMatrix
    tmp = clone(preprocessor)
    X_tr = tmp.fit_transform(X_train_full, y_train)
    X_va = tmp.transform(X_val_full)

    dtrain = xgb.DMatrix(X_tr, label=y_train)
    dvalid = xgb.DMatrix(X_va, label=y_val)

    # 4) Callback de pruning
    pruning_cb = XGBoostPruningCallback(trial, 'validation_0-mae')

    # 5) Entrenamiento con xgb.train
    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=trial.suggest_int('xgb_n_estimators', 50, 1000),
        evals=[(dvalid, 'validation_0')],
        callbacks=[pruning_cb],
        verbose_eval=False
    )

    # 6) Predicción y métrica
    preds = booster.predict(dvalid)
    return mean_absolute_error(y_val, preds)


# --- Configurar y ejecutar el estudio ---
sampler = TPESampler(seed=random_seed)
study = optuna.create_study(direction='minimize', sampler=sampler)
timeout = 300  # segundos

print(f"Iniciando optimización con pruning por {timeout} segundos...")
start = time.time()
study.optimize(objective_with_pruning, timeout=timeout)
end = time.time()

print(f"Optimización completada en {end - start:.2f}s")
print(f"Trials completados: {len(study.trials)}, Mejor MAE: {study.best_value:.2f}")
print("Mejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


Iniciando optimización con pruning por 300 segundos...
Optimización completada en 300.36s
Trials completados: 779, Mejor MAE: 1895.56
Mejores hiperparámetros:
  ohe_min_frequency: 0.06776216245386778
  xgb_learning_rate: 0.092986082404241
  xgb_max_depth: 9
  xgb_max_leaves: 94
  xgb_min_child_weight: 3
  xgb_reg_alpha: 0.8722683476686132
  xgb_reg_lambda: 0.5319814793099852
  xgb_n_estimators: 870




1.  **Impacto de la Poda en los Trials**: Comparando los **779 trials** completados aquí con los **237 trials** de la sección anterior (Optuna sin poda activa) en el mismo tiempo (5 minutos), resulta en un aumento de más del triple. Esto demuestra claramente la **eficiencia ganada gracias a la poda**: Optuna pudo descartar rápidamente los trials malos y explorar muchas más combinaciones de hiperparámetros.
2.  **Impacto de la Poda en el Error (MAE)**: El mejor MAE se redujo de **1931.11** (sin poda) a **1895.56** (con poda). Es una mejora modesta (unos ~35.6 puntos de MAE), pero es una mejora neta.
3.  **¿A qué se debe el cambio?**: La mejora en el MAE, aunque pequeña, probablemente se deba a que la poda permitió una **exploración más amplia y eficiente** del espacio de hiperparámetros. Con más de 3 veces más trials, Optuna tuvo más oportunidades de refinar la búsqueda y encontrar una combinación ligeramente superior, especialmente ajustando los parámetros de regularización (`reg_alpha` es notablemente más alto, `reg_lambda` más bajo) y el número de estimadores.


In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import xgboost as xgb

# 1. Obtener los mejores hiperparámetros
best_params_pruning = study.best_params 
best_ohe_freq = best_params_pruning['ohe_min_frequency']
best_xgb_params = {
    'random_state': random_seed,
    'learning_rate': best_params_pruning['xgb_learning_rate'],
    'n_estimators': best_params_pruning['xgb_n_estimators'], 
    'max_depth': best_params_pruning['xgb_max_depth'],
    'max_leaves': best_params_pruning['xgb_max_leaves'],
    'min_child_weight': best_params_pruning['xgb_min_child_weight'],
    'reg_alpha': best_params_pruning['xgb_reg_alpha'],
    'reg_lambda': best_params_pruning['xgb_reg_lambda'],
    'objective': 'reg:squarederror',
    'n_jobs': -1
}

# 2. Reconstruir el preprocesador con la mejor frecuencia de OHE
final_date_pipeline = Pipeline([
    ('extract_date_features', date_transformer_corrected),
    ('ohe_date_features', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=best_ohe_freq))
])
final_preprocessor = ColumnTransformer(
    transformers=[
        ('date_processing', final_date_pipeline, ['date']),
        ('categorical_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=best_ohe_freq), categorical_cols_original),
        ('numerical_scaling', StandardScaler(), numerical_cols)
    ],
    remainder='drop'
)

# 3. Crear la instancia final del modelo XGBRegressor
final_xgb_model = xgb.XGBRegressor(**best_xgb_params)

# 4. Crear el pipeline final completo
final_pipeline_optuna_pruning = Pipeline([
    ('preprocessor', final_preprocessor),
    ('regressor', final_xgb_model)
])

# 5. Entrenar el pipeline final en TODO el conjunto de entrenamiento
print("\nEntrenando el pipeline final con los mejores parámetros encontrados (con poda)...")
final_pipeline_optuna_pruning.fit(X_train_full, y_train)
print("Pipeline final entrenado.")

# 6. Guardar el pipeline final entrenado
path_optuna_pruning_best_model = 'xgb_model_optuna_pruning_best.pkl'
print(f"\nGuardando el pipeline final en {path_optuna_pruning_best_model}...")
joblib.dump(final_pipeline_optuna_pruning, path_optuna_pruning_best_model)
print(f"Pipeline final guardado en {path_optuna_pruning_best_model}.")


Entrenando el pipeline final con los mejores parámetros encontrados (con poda)...
Pipeline final entrenado.

Guardando el pipeline final en xgb_model_optuna_pruning_best.pkl...
Pipeline final guardado en xgb_model_optuna_pruning_best.pkl.


## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
import optuna
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_param_importances

# --- Generar Gráficos ---

# 1. Gráfico de Historial de Optimización
print("Generando Gráfico de Historial de Optimización...")
fig_history = plot_optimization_history(study)
fig_history.show() 

# 2. Gráfico de Coordenadas Paralelas
print("\nGenerando Gráfico de Coordenadas Paralelas...")
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

# 3. Gráfico de Importancia de Hiperparámetros
print("\nGenerando Gráfico de Importancia de Hiperparámetros...")

try:
    fig_importance = plot_param_importances(study)
    fig_importance.show()

except Exception as e:
    print(f"\nNo se pudo generar el gráfico de importancia de parámetros. Error: {e}")


Generando Gráfico de Historial de Optimización...



Generando Gráfico de Coordenadas Paralelas...



Generando Gráfico de Importancia de Hiperparámetros...


**Respuestas a las Preguntas:**

4.  **¿Desde qué *trial* se empiezan a observar mejoras notables?**
    * Viendo el gráfico de historial, las mejoras más significativas ocurren muy temprano. El mejor MAE (línea roja) baja drásticamente **dentro de los primeros ~50 trials**, alcanzando ya un valor cercano al óptimo final (alrededor de 2000). Después de eso, las mejoras son mucho más graduales.

5.  **¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas?**
    * Los mejores resultados (MAE más bajos, líneas azul oscuro) tienden a asociarse con:
        * Valores altos de `n_estimators` (>~600).
        * Valores altos de `max_depth` (7-10).
        * Valores altos de `max_leaves` (>~60).
        * Valores relativamente altos de `learning_rate` (cercanos a 0.1).
    * No hay tendencias tan claras para `ohe_min_frequency` (aunque el óptimo fue bajo), `min_child_weight`, `reg_alpha` o `reg_lambda` solo mirando las líneas, aunque los valores óptimos encontrados dan una indicación.

6.  **¿Cuáles son los hiperparámetros con mayor importancia?**
    * Según el gráfico de importancia, **`ohe_min_frequency`** es, con diferencia, el hiperparámetro más importante.
    * Le siguen, a mucha distancia, `xgb_learning_rate` y `xgb_reg_lambda`.
    * Los demás hiperparámetros de XGBoost optimizados (`max_depth`, `n_estimators`, `max_leaves`, `min_child_weight`, `reg_alpha`) mostraron tener una importancia mucho menor en esta optimización específica. Esto es un poco sorprendente para parámetros como `n_estimators` o `max_depth`, pero la importancia aquí se refiere a cuánto influyeron en la *variación* del MAE *dentro de los rangos explorados* y en combinación con los otros parámetros. La alta importancia de `ohe_min_frequency` resalta lo crítico que puede ser el preprocesamiento de características categóricas.


## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
import pandas as pd

# Datos para la tabla resumen
data_summary = {
    'Modelo': [
        '1. Baseline - DummyRegressor',
        '2. Baseline - XGBoost (Default)',
        '3. XGBoost Monotónico (Precio)',
        '4. XGBoost Optuna (Sin Poda)',
        '5. XGBoost Optuna (Con Poda)'
    ],
    'MAE en Validación': [
        13543.96,
        2404.40,
        2524.87,
        1931.11,
        1895.56 
    ]
}

# Crear DataFrame
df_summary = pd.DataFrame(data_summary)

# Ordenar por MAE 
df_summary = df_summary.sort_values(by='MAE en Validación', ascending=True)

# Mostrar la tabla
print("Tabla Resumen: MAE en el Conjunto de Validación")
print(df_summary.to_markdown(index=False)) 

Tabla Resumen: MAE en el Conjunto de Validación
| Modelo                          |   MAE en Validación |
|:--------------------------------|--------------------:|
| 5. XGBoost Optuna (Con Poda)    |             1895.56 |
| 4. XGBoost Optuna (Sin Poda)    |             1931.11 |
| 2. Baseline - XGBoost (Default) |             2404.4  |
| 3. XGBoost Monotónico (Precio)  |             2524.87 |
| 1. Baseline - DummyRegressor    |            13544    |


**2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento?**

El **Modelo 5: XGBoost Optimizado con Optuna (con poda activa - xgb.train)** obtuvo el MAE más bajo (**1895.56**). Este es el mejor modelo hasta ahora, según la evaluación en el conjunto de validación.

In [ ]:
import joblib
from sklearn.metrics import mean_absolute_error

# Nombre del archivo del mejor modelo guardado
path_best_model = 'xgb_model_optuna_pruning_best.pkl' 

try:
    # Cargar el mejor modelo (pipeline entrenado)
    print(f"\nCargando el mejor modelo desde {path_best_model}...")
    best_model_loaded = joblib.load(path_best_model)
    print("Modelo cargado exitosamente.")

    # Realizar predicciones en el conjunto de TEST
    print("\nRealizando predicciones en el conjunto de test...")
    y_pred_test = best_model_loaded.predict(X_test_full)

    # Calcular el MAE en el conjunto de TEST
    mae_test = mean_absolute_error(y_test, y_pred_test)

    print(f"\nMAE del mejor modelo en el CONJUNTO DE TEST: {mae_test:.2f}")

    best_mae_validation = 1895.56 # El mejor MAE en validación

except FileNotFoundError:
    print(f"\nError: No se encontró el archivo del modelo '{path_best_model}'.")
except Exception as e:
    print(f"\nOcurrió un error al cargar o usar el modelo: {e}")


Cargando el mejor modelo desde xgb_model_optuna_pruning_best.pkl...
Modelo cargado exitosamente.

Realizando predicciones en el conjunto de test...

MAE del mejor modelo en el CONJUNTO DE TEST: 1886.84


**4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto?**

**Comparación:**
* MAE en Validación (del mejor modelo): 1895.56
* MAE en Test (del mejor modelo): **1886.84**

Se observa que el MAE en el conjunto de test es **muy similar** al MAE obtenido en el conjunto de validación. De hecho, es incluso ligeramente *mejor* (más bajo) en el test set. La diferencia es mínima (aproximadamente 8.7 unidades de MAE).

**Análisis de la Diferencia (o falta de ella):**
Este es un resultado **muy positivo**. Indica que:
1.  **Buena Generalización**: El modelo final, con los hiperparámetros encontrados por Optuna (con poda activa), generaliza muy bien a datos completamente nuevos que no se utilizaron durante el entrenamiento ni la optimización.
2.  **No Hay Sobreajuste (a Validación)**: El proceso de optimización no parece haber sobreajustado el modelo al conjunto de validación específico. El rendimiento se mantiene (o incluso mejora ligeramente) en el conjunto de test.
3.  **División de Datos Representativa**: Sugiere que la división original en conjuntos de entrenamiento, validación y test fue razonablemente buena, y que estos conjuntos tienen características similares.

**¿Por qué la pequeña diferencia (Test mejor que Validación)?**
Aunque a menudo se espera que el rendimiento en test sea igual o ligeramente peor que en validación, un rendimiento ligeramente mejor en test es posible y puede deberse a:
* **Variabilidad Aleatoria**: Simplemente por azar, la composición específica de las muestras en el conjunto de test podría hacerlo marginalmente "más fácil" de predecir para este modelo en particular en comparación con el conjunto de validación.
* **Tamaño del Conjunto**: A veces, si el conjunto de test es pequeño, la métrica puede tener un poco más de ruido. Sin embargo, la consistencia aquí es la característica clave.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=87110296-876e-426f-b91d-aaf681223468' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>